# 03 Retention-Aware MBA  ·  *Finals cohort*

**Client deliverable for LushProtein — Customer Retention via Market Basket Analysis**

> **Data note:** this notebook runs on the **finals analysis pool** (`EDA/outputs_finals/`) — the post-presentation clean cohort: DQ-filtered orders, finals-eligible customers (2022+, excl. July/Nov acquisitions, excl. elite, excl. 51%+ first-order discounts), and the `lines_sku_analysis` line subset. It is smaller and cleaner than the midterm base, so absolute numbers differ; the *method* and the *story* are unchanged.

Notebook `02` ranked product pairs by **lift** — *"what raises basket size today?"* (an AOV question). This notebook keeps that classic MBA as the foundation but **re-scores every rule by its retention impact**, because in this business cross-sell is fundamentally a *retention* lever, not just an AOV one. We rebuild the two anchoring facts (the breadth cliff and the subscription gap) live from the finals customer base, then build a composite **Retention Value Score** and surface **retention-anchor products**.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "EDA" / "outputs").exists() and (candidate / "product_mba").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing EDA/outputs and product_mba")

PROJECT_ROOT = find_project_root()
EDA = PROJECT_ROOT / "EDA" / "outputs_finals"   # FINALS cohort
MBA_OUTPUTS = PROJECT_ROOT / "product_mba" / "outputs"
MBA_OUTPUTS.mkdir(exist_ok=True)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
print("Finals cohort:", EDA.name)

Finals cohort: outputs_finals


In [2]:
# --- Business baselines from the finals customer base ---
customers = pd.read_parquet(EDA / "customers.parquet")
customers["customer_id"] = customers["customer_id"].astype(str)

BASE_REPEAT = customers["is_repeat"].mean()
BASE_SUB = customers["ever_subscribed"].mean()
BASE_LTV = customers["total_revenue"].mean()

# Recompute the breadth cliff directly from finals data (no midterm CSV dependency):
# distinct product categories a customer bought across the finals SKU-analysis lines.
lines = pd.read_parquet(EDA / "lines_sku_analysis.parquet")
lines["customer_id"] = lines["customer_id"].astype(str)
ncat = (lines.dropna(subset=["product_category"])
        .groupby("customer_id")["product_category"].nunique())
cb = customers[["customer_id", "is_repeat", "total_revenue", "total_orders"]].copy()
cb["n_cat"] = cb["customer_id"].map(ncat).fillna(0).astype(int)
cb = cb[cb["n_cat"] >= 1]
cb["band"] = np.where(cb["n_cat"] >= 4, "4+ products", cb["n_cat"].astype(str) + " product"
                      + np.where(cb["n_cat"] == 1, "", "s"))
order = ["1 product", "2 products", "3 products", "4+ products"]
breadth = (cb.groupby("band")
           .agg(customers=("customer_id", "size"), repeat_rate=("is_repeat", "mean"),
                avg_ltv=("total_revenue", "mean"), avg_orders=("total_orders", "mean"))
           .reindex(order).dropna(how="all").reset_index().rename(columns={"index": "band"}))
breadth.to_csv(MBA_OUTPUTS / "finals_breadth_cliff.csv", index=False)

print(f"Base repeat rate {BASE_REPEAT:.1%} | subscription {BASE_SUB:.1%} | "
      f"avg LTV ${BASE_LTV:,.0f} | customers {len(customers):,}")
print("\nThe breadth cliff (cross-sell as a retention lever) — finals cohort:")
display(breadth)

Base repeat rate 22.9% | subscription 11.2% | avg LTV $129 | customers 6,353

The breadth cliff (cross-sell as a retention lever) — finals cohort:


,band,customers,repeat_rate,avg_ltv,avg_orders
0,1 product,3681,0.160,105.239,1.281
1,2 products,1411,0.333,145.841,1.822
2,3 products,445,0.587,247.853,2.620
3,4+ products,157,0.841,742.832,8.866


In [3]:
# --- Re-score every 02 rule by retention impact, not raw lift ---
MIN_BOTH_CUSTOMERS = 30

def minmax(s):
    s = s.astype(float)
    lo, hi = s.min(), s.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(0.5, index=s.index)
    return (s - lo) / (hi - lo)

def retention_score(level):
    df = pd.read_csv(MBA_OUTPUTS / f"mba_rules_{level}.csv")
    if df.empty:
        return df
    df["pair_est_margin_pct"] = df[["antecedent_item_est_margin_pct",
                                    "consequent_item_est_margin_pct"]].mean(axis=1)
    df["repeat_rate_uplift"] = df["both_item_repeat_rate"] - BASE_REPEAT
    df["subscription_uplift"] = df["both_item_pct_subscribed"] - BASE_SUB
    df["ltv_ratio"] = df["both_item_avg_ltv"] / BASE_LTV
    df["trustworthy"] = df["both_item_customers"] >= MIN_BOTH_CUSTOMERS
    scored = df[df["trustworthy"]].copy()
    if scored.empty:
        scored = df.copy()
    scored["n_repeat_uplift"] = minmax(scored["repeat_rate_uplift"])
    scored["n_sub_uplift"] = minmax(scored["subscription_uplift"])
    scored["n_lift"] = minmax(scored["lift"])
    scored["n_scale"] = minmax(np.log1p(scored["co_orders"]))
    scored["n_margin"] = minmax(scored["pair_est_margin_pct"].fillna(scored["pair_est_margin_pct"].median()))
    scored["retention_value_score"] = (
        0.30 * scored["n_repeat_uplift"] + 0.25 * scored["n_sub_uplift"]
        + 0.20 * scored["n_lift"] + 0.15 * scored["n_scale"] + 0.10 * scored["n_margin"])
    scored["level"] = level
    return scored.sort_values("retention_value_score", ascending=False)

show_cols = ["antecedent", "consequent", "co_orders", "lift", "confidence",
             "both_item_repeat_rate", "repeat_rate_uplift", "both_item_pct_subscribed",
             "subscription_uplift", "ltv_ratio", "retention_value_score"]

scored_tables = {}
for level in ["category", "handle", "sku_flavor"]:
    s = retention_score(level)
    scored_tables[level] = s
    s.to_csv(MBA_OUTPUTS / f"mba_rules_retention_scored_{level}.csv", index=False)
    print(f"\n=== {level}: top 8 by Retention Value Score (n={len(s)}) ===")
    display(s[show_cols].head(8).reset_index(drop=True))


=== category: top 8 by Retention Value Score (n=4) ===


,antecedent,consequent,co_orders,lift,confidence,both_item_repeat_rate,repeat_rate_uplift,both_item_pct_subscribed,subscription_uplift,ltv_ratio,retention_value_score
0,Accessories,Clear Protein,341,1.101,0.319,0.457,0.228,0.241,0.129,2.402,0.650
1,Clear Protein,Accessories,341,1.101,0.160,0.457,0.228,0.241,0.129,2.402,0.650
2,Lean Protein,Accessories,382,1.355,0.197,0.409,0.181,0.218,0.106,1.939,0.350
3,Accessories,Lean Protein,382,1.355,0.358,0.409,0.181,0.218,0.106,1.939,0.350



=== handle: top 8 by Retention Value Score (n=11) ===


,antecedent,consequent,co_orders,lift,confidence,both_item_repeat_rate,repeat_rate_uplift,both_item_pct_subscribed,subscription_uplift,ltv_ratio,retention_value_score
0,pureburn-fat-burner-capsules,green-tea-extract-capsules,29,23.349,0.232,0.600,0.371,0.200,0.088,13.166,0.700
1,green-tea-extract-capsules,pureburn-fat-burner-capsules,29,23.349,0.397,0.600,0.371,0.200,0.088,13.166,0.700
2,lushprotein-clear-shaker,lean-protein,277,1.205,0.259,0.458,0.230,0.262,0.149,2.261,0.556
3,lean-protein,lushprotein-clear-shaker,277,1.205,0.175,0.458,0.230,0.262,0.149,2.261,0.556
4,lean-protein-peach-oolong-pre-order,lushprotein-clear-shaker,30,2.428,0.353,0.568,0.339,0.205,0.092,1.767,0.451
5,lushprotein-lean-protein-40g-single-serve,lushprotein-clear-shaker,91,1.944,0.283,0.443,0.215,0.165,0.053,4.564,0.422
6,lushprotein-clear-shaker,lushprotein-lean-protein-40g-single-serve,91,1.944,0.085,0.443,0.215,0.165,0.053,4.564,0.422
7,discovery-sampler,lushprotein-clear-shaker,40,1.456,0.212,0.436,0.207,0.154,0.041,11.641,0.338



=== sku_flavor: top 8 by Retention Value Score (n=66) ===


,antecedent,consequent,co_orders,lift,confidence,both_item_repeat_rate,repeat_rate_uplift,both_item_pct_subscribed,subscription_uplift,ltv_ratio,retention_value_score
0,LEAN-THA-1KG-V1 | LEAN PROTEIN | 1 x 1kg Pack ...,LEAN-TAR-1KG-V1 | LEAN PROTEIN | 1 x 1kg Pack ...,48,4.172,0.143,0.522,0.293,0.435,0.322,9.176,0.563
1,LEAN-TAR-1KG-V1 | LEAN PROTEIN | 1 x 1kg Pack ...,LEAN-THA-1KG-V1 | LEAN PROTEIN | 1 x 1kg Pack ...,48,4.172,0.185,0.522,0.293,0.435,0.322,9.176,0.563
2,0724999808361 | [FREE] LushProtein Classic Sha...,0724999807814 | CLEAR PROTEIN | 500g Pack (20 ...,40,4.922,0.426,0.561,0.332,0.415,0.302,2.031,0.562
3,0724999807814 | CLEAR PROTEIN | 500g Pack (20 ...,0724999808361 | [FREE] LushProtein Classic Sha...,40,4.922,0.061,0.561,0.332,0.415,0.302,2.031,0.562
4,0724999807821 | LEAN PROTEIN | 1kg Pack (25 se...,0724999807814 | CLEAR PROTEIN | 500g Pack (20 ...,45,1.839,0.159,0.632,0.403,0.298,0.186,8.566,0.540
5,0724999807814 | CLEAR PROTEIN | 500g Pack (20 ...,0724999807821 | LEAN PROTEIN | 1kg Pack (25 se...,45,1.839,0.069,0.632,0.403,0.298,0.186,8.566,0.540
6,0724999807807 | CLEAR PROTEIN | 500g Pack (20 ...,0724999807821 | LEAN PROTEIN | 1kg Pack (25 se...,31,1.954,0.073,0.667,0.438,0.310,0.197,10.514,0.539
7,0724999807821 | LEAN PROTEIN | 1kg Pack (25 se...,0724999807807 | CLEAR PROTEIN | 500g Pack (20 ...,31,1.954,0.110,0.667,0.438,0.310,0.197,10.514,0.539


In [4]:
# --- AOV-optimal vs retention-optimal: do the rankings differ? ---
def top_pairs(df, by, k=10):
    d = df.sort_values(by, ascending=False).head(k)
    return set(zip(d["antecedent"], d["consequent"]))

for level in ["category", "handle"]:
    s = scored_tables[level]
    if s.empty:
        continue
    lift_top, ret_top = top_pairs(s, "lift"), top_pairs(s, "retention_value_score")
    only_ret = ret_top - lift_top
    print(f"[{level}] top-10 overlap between lift and retention ranking: {len(lift_top & ret_top)}/{min(10,len(s))}")
    for a, b in list(only_ret)[:6]:
        print(f"    retention promotes (lift misses):  {a}  ->  {b}")

[category] top-10 overlap between lift and retention ranking: 4/4
[handle] top-10 overlap between lift and retention ranking: 9/10
    retention promotes (lift misses):  lean-protein  ->  lushprotein-clear-shaker


In [5]:
# --- Retention-anchor products ---
def anchor_table(level):
    s = scored_tables[level]
    if s.empty:
        return pd.DataFrame()
    long = pd.concat([
        s[["antecedent", "co_orders", "repeat_rate_uplift", "subscription_uplift", "ltv_ratio"]].rename(columns={"antecedent": "item"}),
        s[["consequent", "co_orders", "repeat_rate_uplift", "subscription_uplift", "ltv_ratio"]].rename(columns={"consequent": "item"}),
    ], ignore_index=True)
    def wavg(g, col):
        w = g["co_orders"]
        return np.average(g[col], weights=w) if w.sum() else np.nan
    rows = []
    for item, g in long.groupby("item"):
        rows.append({"level": level, "item": item, "appears_in_rules": len(g),
                     "total_co_orders": int(g["co_orders"].sum()),
                     "wtd_repeat_uplift": wavg(g, "repeat_rate_uplift"),
                     "wtd_subscription_uplift": wavg(g, "subscription_uplift"),
                     "wtd_ltv_ratio": wavg(g, "ltv_ratio")})
    out = pd.DataFrame(rows)
    out["anchor_score"] = (minmax(out["wtd_repeat_uplift"]) + minmax(out["wtd_subscription_uplift"])) / 2
    return out.sort_values("anchor_score", ascending=False)

anchors = pd.concat([anchor_table("category"), anchor_table("handle")], ignore_index=True)
anchors.to_csv(MBA_OUTPUTS / "retention_anchor_products.csv", index=False)
print("Top retention-anchor products (category + handle), finals cohort:")
display(anchors.head(15).reset_index(drop=True))

Top retention-anchor products (category + handle), finals cohort:


,level,item,appears_in_rules,total_co_orders,wtd_repeat_uplift,wtd_subscription_uplift,wtd_ltv_ratio,anchor_score
0,category,Clear Protein,2,682,0.228,0.129,2.402,1.000
1,category,Accessories,4,1446,0.203,0.117,2.158,0.472
2,category,Lean Protein,2,764,0.181,0.106,1.939,0.000
3,handle,green-tea-extract-capsules,2,58,0.371,0.088,13.166,0.838
4,handle,pureburn-fat-burner-capsules,2,58,0.371,0.088,13.166,0.838
5,handle,lean-protein-peach-oolong-pre-order,1,30,0.339,0.092,1.767,0.783
6,handle,lean-protein,2,554,0.230,0.149,2.261,0.703
7,handle,lushprotein-clear-shaker,7,878,0.226,0.107,3.376,0.583
8,handle,discovery-sampler,1,40,0.207,0.041,11.641,0.372
9,handle,lushprotein-lean-protein-40g-single-serve,4,384,0.162,0.003,4.403,0.176


## Readout: from "biggest basket" to "stickiest basket"

**1. Lift and retention are not the same ranking.** The overlap check shows the retention lens reshuffles `02`'s pure-lift priority list. High-lift sachet/flavor pairs are good *discovery* mechanics but weak retention anchors on their own; cross-category combos that move a customer up the breadth cliff are where repeat rate and LTV compound.

**2. The breadth cliff is the strategy.** The finals breadth table (recomputed above) shows repeat rate and LTV stepping up with each additional category bought. Every cross-sell rule should be judged on whether it *advances a customer across that cliff* — exactly what `repeat_rate_uplift` and `ltv_ratio` now measure per rule.

**3. Subscription is the retention jackpot — score for it.** `subscription_uplift` is weighted heavily; pairs that over-index on subscribers (vs the finals baseline) are the combos to convert into subscribe-and-save in `05`.

**4. Retention-anchor products are the merchandising spine.** `retention_anchor_products.csv` ranks items by how much *every* basket containing them lifts repeat/subscription — the products to feature regardless of raw sales volume.

**Caveat:** margin is directional only (`00_margin_data.ipynb`); it is the lowest-weighted term and a tie-breaker, never the driver.

Outputs: `mba_rules_retention_scored_{level}.csv`, `retention_anchor_products.csv`, `finals_breadth_cliff.csv`.

---
### Final metrics & scores  ·  *finals cohort*

**Baselines:** repeat rate **23%** · subscription **11%** · avg LTV **$129**. Rules scored: category **4** · handle **11** · SKU/flavor **66** (≥30 both-item customers).

**Breadth cliff (finals):** 1→2→3→4+ categories = repeat **16% → 33% → 59% → 84%**, LTV **$105 → $146 → $248 → $743**.

**Top rules by Retention Value Score (RVS)** — uplift = points above baseline:

| Level | Rule | RVS | Lift | Repeat uplift | Sub. uplift | LTV× |
|---|---|--:|--:|--:|--:|--:|
| Handle | green-tea-extract ↔ pureburn | 0.70 | 23.3 | +37 pts | +9 pts | 13.2× |
| Handle | clear-shaker ↔ lean-protein | 0.56 | 1.20 | +23 pts | +15 pts | 2.3× |
| Category | Accessories ↔ Clear Protein | 0.65 | 1.10 | +23 pts | +13 pts | 2.4× |
| Category | Accessories ↔ Lean Protein | 0.35 | 1.36 | +18 pts | +11 pts | 1.9× |

**Top retention-anchor products** (handle level, by `anchor_score`):

| Product | Anchor score | Repeat uplift | Sub. uplift | LTV× |
|---|--:|--:|--:|--:|
| green-tea-extract / pureburn | 0.84 | +37 pts | +9 pts | 13.2× |
| lean-protein | 0.70 | +23 pts | +15 pts | 2.3× |
| lushprotein-clear-shaker | 0.58 | +23 pts | +11 pts | 3.4× |
| discovery-sampler | ~0.5 | +21 pts | +4 pts | 11.6× |

**Read:** the ranking reshuffles versus `02` — pure-lift Accessories pairs sit mid-table while **supplement attaches and breadth-builders** (huge repeat / subscription / LTV uplift) rise to the top. Score for stickiness, not basket size.
